## 1. Load Analytical Dataset into Pandas

DuckDB was used in the previous phase for large-scale inspection of the raw Parquet dataset.

For cleaning and exploratory analysis, we now move only the transaction-level analytical columns into Pandas. Raw identifiers and high-cardinality entity fields are kept outside the main analytical DataFrame until their potential use in feature engineering is justified.

In [1]:
import duckdb
import pandas as pd

In [2]:
# Raw dataset
DATA_PATH = "../data/raw/airline_fraud_3M.parquet"

# Columns required for cleaning and EDA
ANALYTICAL_COLUMNS = [
    "transaction_date",
    "route",
    "amount",
    "amount_in_usd",
    "currency",
    "card_type",
    "card_bin",
    "bank_name",
    "bin_country",
    "loyalty_tier",
    "txn_origin_country",
    "account_age_days",
    "failed_attempts",
    "is_fraud",
    "billing_country",
    "session_time_seconds"
]

query = f"""
SELECT {", ".join(ANALYTICAL_COLUMNS)}
FROM read_parquet('{DATA_PATH}')
"""

df = duckdb.sql(query).df()
df.shape

(3000000, 16)

## 2. Check Pandas Memory Usage

Before further analysis, check the memory footprint of the working DataFrame. This helps determine whether the current representation is practical for downstream cleaning and EDA.

In [3]:
df.memory_usage(deep=True).sort_values(ascending=False)

route                   45000000
bank_name               43643221
card_type               43199712
card_bin                42000000
loyalty_tier            39302306
currency                33000000
billing_country         30000000
bin_country             30000000
txn_origin_country      30000000
account_age_days        24000000
amount                  24000000
amount_in_usd           24000000
transaction_date        24000000
is_fraud                24000000
failed_attempts         24000000
session_time_seconds    12000000
Index                        132
dtype: int64

In [ ]:
df.memory_usage(deep=True).sum() / (1024**2)

np.float64(469.3464002609253)

## 3. Data Quality Inventory

Summarize missingness and cardinality across the working dataset before making any transformations. This provides the baseline for cleaning decisions.

In [5]:
quality_summary = pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100),
    "unique": df.nunique(dropna=False)
}).sort_values("missing_pct", ascending=False)

quality_summary

,dtype,missing,missing_pct,unique
transaction_date,datetime64[us],0,0.0,2727067
route,str,0,0.0,1558
amount,float64,0,0.0,328473
amount_in_usd,float64,0,0.0,327626
currency,str,0,0.0,7
card_type,str,0,0.0,3
card_bin,str,0,0.0,24
bank_name,str,0,0.0,8
bin_country,str,0,0.0,4
loyalty_tier,str,0,0.0,4


## 4. Duplicate Records

Check for exact duplicates across the analytical columns. Duplicate rows will be investigated before any removal decision is made.

In [6]:
duplicate_count = df.duplicated().sum()

print(f"Exact duplicate rows: {duplicate_count:,}")
print(f"Duplicate share: {duplicate_count / len(df) * 100:.4f}%")

Exact duplicate rows: 0
Duplicate share: 0.0000%


No exact duplicate records were found across the analytical columns, so no duplicate rows were removed.😁

## 5. Categorical Value Inspection

Inspect the distinct values of low-cardinality categorical variables to identify inconsistent labels or unexpected categories before cleaning.

In [7]:
# Automatically grab all object and category columns
categorical_df = df.select_dtypes(include=["object", "string"])

categorical_quality = pd.DataFrame({
    "unique": categorical_df.nunique(dropna=False),
    "blank": categorical_df.apply(
        lambda s: s.str.strip().eq("").sum()
    ),
    "leading_trailing_space": categorical_df.apply(
        lambda s: s.ne(s.str.strip()).sum()
    ),
    "lowercase_variants": categorical_df.apply(
        lambda s: s.str.lower().nunique(dropna=False)
    )
})

categorical_quality

,unique,blank,leading_trailing_space,lowercase_variants
route,1558,0,0,1558
currency,7,0,0,7
card_type,3,0,0,3
card_bin,24,0,0,24
bank_name,8,0,0,8
bin_country,4,0,0,4
loyalty_tier,4,0,0,4
txn_origin_country,19,0,0,19
billing_country,4,0,0,4


## 6. Memory Optimization Check

Evaluate whether categorical encoding can reduce the memory footprint of string-based columns before deeper EDA.

In [8]:
memory_comparison = pd.DataFrame({
    "current_mb": categorical_df.memory_usage(deep=True).iloc[1:] / (1024**2),
    "category_mb": categorical_df.astype("category").memory_usage(deep=True).iloc[1:] / (1024**2)
})

memory_comparison["saving_pct"] = (
    (memory_comparison["current_mb"] - memory_comparison["category_mb"])
    / memory_comparison["current_mb"]
    * 100
)

memory_comparison.sort_values("current_mb", ascending=False)

,current_mb,category_mb,saving_pct
route,42.915344,5.744519,86.614300
bank_name,41.621419,2.861133,93.125817
card_type,41.198456,2.861064,93.055410
card_bin,40.054321,2.861346,92.856336
loyalty_tier,37.481600,2.861075,92.366720
currency,31.471252,2.861097,90.908855
bin_country,28.610229,2.861062,89.999863
txn_origin_country,28.610229,2.861207,89.999357
billing_country,28.610229,2.861062,89.999863


### 6.1 Numeric Memory Optimization

Check the ranges of integer columns before reducing their storage type.

In [9]:
integer_df = df.select_dtypes(include=["integer"])

integer_range = pd.DataFrame({
    "dtype": integer_df.dtypes,
    "min": integer_df.min(),
    "max": integer_df.max()
})

integer_range

,dtype,min,max
account_age_days,int64,0,3712
failed_attempts,int64,0,5
is_fraud,int64,0,1
session_time_seconds,int32,45,1199


### 6.2 Apply Memory Optimization

Convert categorical variables to `category` and reduce integer storage types where their observed ranges safely fit.

In [10]:
# Categorical columns
categorical_df = df.select_dtypes(include=["object", "string"])

for col in categorical_df.columns:
    df[col] = df[col].astype("category")

# Integer columns
df["account_age_days"] = df["account_age_days"].astype("int16")
df["failed_attempts"] = df["failed_attempts"].astype("int8")
df["is_fraud"] = df["is_fraud"].astype("int8")
df["session_time_seconds"] = df["session_time_seconds"].astype("int16")

df.memory_usage(deep=True).sum() / (1024**2)

np.float64(114.46438026428223)

In [11]:
# Convert bytes to MB for every column individually
memory_in_mb = df.memory_usage(deep=True) / (1024 ** 2)
memory_in_mb.sort_values(ascending=False)

transaction_date        22.888184
amount_in_usd           22.888184
amount                  22.888184
route                    5.744519
session_time_seconds     5.722046
account_age_days         5.722046
card_bin                 2.861346
txn_origin_country       2.861207
bank_name                2.861133
currency                 2.861097
loyalty_tier             2.861075
card_type                2.861064
bin_country              2.861062
billing_country          2.861062
is_fraud                 2.861023
failed_attempts          2.861023
Index                    0.000126
dtype: float64

## 7. Numeric Validity Checks

Check for values that violate basic logical constraints before investigating distributions and potential outliers.

In [12]:
numeric_checks = {
    "amount < 0": (df["amount"] < 0).sum(),
    "amount_in_usd < 0": (df["amount_in_usd"] < 0).sum(),
    "account_age_days < 0": (df["account_age_days"] < 0).sum(),
    "failed_attempts < 0": (df["failed_attempts"] < 0).sum(),
    "session_time_seconds <= 0": (df["session_time_seconds"] <= 0).sum(),
    "is_fraud not in {0, 1}": (~df["is_fraud"].isin([0, 1])).sum()
}

pd.Series(numeric_checks, name="invalid_rows")

amount < 0                   0
amount_in_usd < 0            0
account_age_days < 0         0
failed_attempts < 0          0
session_time_seconds <= 0    0
is_fraud not in {0, 1}       0
Name: invalid_rows, dtype: int64

## 8. Monetary Variables

Examine the relationship between transaction amount and its USD equivalent to determine whether both variables provide distinct information.

In [13]:
amount_relationship = pd.DataFrame({
    "amount": df["amount"],
    "amount_in_usd": df["amount_in_usd"]
})

amount_relationship.corr()

,amount,amount_in_usd
amount,1.000000,0.807035
amount_in_usd,0.807035,1.000000


### 8.1 Amount Conversion by Currency

Compare the empirical USD-to-original-currency ratio across currencies to understand why the two monetary variables are not perfectly correlated.

In [14]:
currency_conversion = (
    df.assign(
        usd_per_unit=df["amount_in_usd"] / df["amount"]
    )
    .groupby("currency", observed=True)["usd_per_unit"]
    .agg(["count", "mean", "median", "min", "max"])
    .sort_values("mean")
)

currency_conversion

,count,mean,median,min,max
currency,,,,,
JPY,300634,0.007,0.007,0.004673,0.010000
INR,149330,0.012,0.012,0.008197,0.015873
AUD,150329,0.650,0.650,0.647059,0.654545
CAD,300011,0.730,0.730,0.725490,0.733945
USD,1200845,1.000,1.000,1.000000,1.000000
EUR,598714,1.150,1.150,1.146341,1.153846
GBP,300137,1.300,1.300,1.296552,1.304762


### 8.2 Currency Conversion Variability

Measure the variability of the implied USD conversion rate within each currency to determine whether the conversion relationship is stable.

In [15]:
conversion_variability = (
    df.assign(
        usd_per_unit=df["amount_in_usd"] / df["amount"]
    )
    .groupby("currency", observed=True)["usd_per_unit"]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .sort_values("std", ascending=False)
)

conversion_variability

,count,mean,std,min,median,max
currency,,,,,,
INR,149330,0.012,0.000044,0.008197,0.012,0.015873
CAD,300011,0.730,0.000041,0.725490,0.730,0.733945
EUR,598714,1.150,0.000039,1.146341,1.150,1.153846
AUD,150329,0.650,0.000038,0.647059,0.650,0.654545
JPY,300634,0.007,0.000037,0.004673,0.007,0.010000
GBP,300137,1.300,0.000036,1.296552,1.300,1.304762
USD,1200845,1.000,0.000000,1.000000,1.000,1.000000


### 8.3 Reconstructing the USD Amount

Test whether the USD amount can be closely reconstructed from the original amount and the currency-specific conversion rate.

In [16]:
currency_rates = (
    df.assign(
        usd_per_unit=df["amount_in_usd"] / df["amount"]
    )
    .groupby("currency", observed=True)["usd_per_unit"]
    .median()
)

reconstructed_usd = df["amount"] * df["currency"].map(currency_rates).astype(float)
reconstruction_error = (df["amount_in_usd"] - reconstructed_usd).abs()

pd.Series({
    "mean_abs_error": reconstruction_error.mean(),
    "median_abs_error": reconstruction_error.median(),
    "max_abs_error": reconstruction_error.max(),
    "exact_match_pct": (reconstruction_error == 0).mean() * 100
})

mean_abs_error       0.001500
median_abs_error     0.001000
max_abs_error        0.005054
exact_match_pct     41.450400
dtype: float64

## 9. Fraud Rate by Transaction Origin

Examine fraud rates alongside transaction volume to distinguish meaningful geographic patterns from small-sample effects.

In [17]:
origin_fraud = (
    df.groupby("txn_origin_country", observed=True)["is_fraud"]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "count": "transactions",
        "sum": "fraud_count",
        "mean": "fraud_rate"
    })
    .sort_values("fraud_rate", ascending=False)
)

origin_fraud["fraud_rate"] = origin_fraud["fraud_rate"] * 100
origin_fraud

,transactions,fraud_count,fraud_rate
txn_origin_country,,,
BR,4952,4952,100.000000
NG,4951,4951,100.000000
CN,5046,5046,100.000000
RU,5051,5051,100.000000
SG,31712,288,0.908174
AU,31362,282,0.899177
DE,31187,279,0.894604
JP,31550,276,0.874802
IN,280830,2442,0.869565


### Fraud Rate by Country and Month

Investigate whether the extreme fraud behavior of selected origin countries persists across the full observation period.

In [18]:
extreme_origins = ["BR", "NG", "CN", "RU", "EG"]

origin_monthly = (
    df[df["txn_origin_country"].isin(extreme_origins)]
    .assign(month=df["transaction_date"].dt.to_period("M"))
    .groupby(["txn_origin_country", "month"], observed=True)["is_fraud"]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "count": "transactions",
        "sum": "fraud_count",
        "mean": "fraud_rate"
    })
)

origin_monthly["fraud_rate"] = origin_monthly["fraud_rate"] * 100
origin_monthly

transactions  fraud_count  fraud_rate
txn_origin_country month                                         
BR                 2025-12           860          860       100.0
                   2026-01           813          813       100.0
                   2026-02           810          810       100.0
                   2026-03           865          865       100.0
                   2026-04           839          839       100.0
                   2026-05           765          765       100.0
CN                 2025-12           909          909       100.0
                   2026-01           859          859       100.0
                   2026-02           815          815       100.0
                   2026-03           831          831       100.0
                   2026-04           852          852       100.0
                   2026-05           780          780       100.0
EG                 2025-12          1376            0         0.0
                   2026-01          1428            0         0.0
                   2026-02          1216            0         0.0
                   2026-03          1433            0         0.0
                   2026-04          1332            0         0.0
                   2026-05          1247            0         0.0
NG                 2025-12           818          818       100.0
                   2026-01           836          836       100.0
                   2026-02           793          793       100.0
                   2026-03           911          911       100.0
                   2026-04           821          821       100.0
                   2026-05           772          772       100.0
RU                 2025-12           845          845       100.0
                   2026-01           902          902       100.0
                   2026-02           798          798       100.0
                   2026-03           848          848       100.0
                   2026-04           844          844       100.0
                   2026-05           814          814       100.0

## 10. Dataset Viability Test

Assess whether meaningful fraud signal remains after excluding the suspicious `txn_origin_country` feature. The test focuses on univariate separation across the remaining variables rather than model performance.

In [19]:
# Exclude the identified synthetic target artifact
viability_df = df.drop(columns="txn_origin_country")

target = viability_df["is_fraud"]

# Categorical signal summary
categorical_cols = viability_df.select_dtypes(include=["category"]).columns

categorical_signal = []

for col in categorical_cols:
    rates = viability_df.groupby(col, observed=True)["is_fraud"].mean()
    categorical_signal.append({
        "feature": col,
        "groups": rates.size,
        "min_fraud_rate_%": rates.min() * 100,
        "median_fraud_rate_%": rates.median() * 100,
        "max_fraud_rate_%": rates.max() * 100
    })

categorical_signal = pd.DataFrame(categorical_signal).sort_values(
    "max_fraud_rate_%", ascending=False
)

# Numeric signal summary using deciles
numeric_cols = viability_df.select_dtypes(include=["number"]).columns.drop("is_fraud")

numeric_signal = []

for col in numeric_cols:
    bins = pd.qcut(viability_df[col], q=10, duplicates="drop")
    rates = viability_df.groupby(bins, observed=True)["is_fraud"].mean()

    numeric_signal.append({
        "feature": col,
        "decile_min_fraud_rate_%": rates.min() * 100,
        "decile_max_fraud_rate_%": rates.max() * 100,
        "decile_range_pp": (rates.max() - rates.min()) * 100
    })

numeric_signal = pd.DataFrame(numeric_signal).sort_values(
    "decile_range_pp", ascending=False
)

print("BASELINE FRAUD RATE:", round(target.mean() * 100, 3), "%")

print("\nCATEGORICAL SIGNAL")
display(categorical_signal)

print("\nNUMERIC SIGNAL")
display(numeric_signal)

BASELINE FRAUD RATE: 1.5 %

CATEGORICAL SIGNAL


,feature,groups,min_fraud_rate_%,median_fraud_rate_%,max_fraud_rate_%
0,route,1558,0.000000,1.492260,8.333333
3,card_bin,24,1.233651,1.554531,3.991732
6,loyalty_tier,4,0.998856,1.645269,1.815747
1,currency,7,1.469650,1.519306,1.546240
4,bank_name,8,1.450639,1.502061,1.531912
5,bin_country,4,1.454290,1.491569,1.515263
7,billing_country,4,1.454290,1.491569,1.515263
2,card_type,3,1.467151,1.501393,1.505465



NUMERIC SIGNAL


,feature,decile_min_fraud_rate_%,decile_max_fraud_rate_%,decile_range_pp
0,amount,0.641677,4.672907,4.031229
1,amount_in_usd,0.616650,3.550864,2.934214
2,account_age_days,0.846570,1.811195,0.964625
4,session_time_seconds,1.458954,1.529193,0.070240
3,failed_attempts,1.500000,1.500000,0.000000


After removing the obvious synthetic artifact, meaningful fraud signal still exists in several independent variables.

Our baseline fraud rate is only 1.5%.

Now look at the remaining variables.

### Stronger signals

* `amount`: `0.64% → 4.67%`
    * That's a **4.03 percentage-point spread** across deciles. That's meaningful.
* `amount_in_usd`: `0.62% → 3.55%`
* `account_age_days`: `0.85% → 1.81%`
* `card_bin`: `1.23% → 3.99%`
* `loyalty_tier`: `~1.00% → 1.82%`
* `route`: `0% → 8.33%`

But with **1,558 routes**, some of those extreme rates could absolutely be small-category effects.

And some variables are basically flat which can actually be useful too.

* `currency`: `1.47% → 1.55%`
* `bank_name`: `1.45% → 1.53%`
* `card_type`: `1.47% → 1.51%`
* `bin_country` / `billing_country`: `1.45% → 1.51%`
    * And since we found `bin_country == billing_country`, one of them will probably disappear eventually anyway.
* `session_time_seconds`: `1.459% → 1.529%`
* `failed_attempts`: `1.5% → 1.5%`

**Zero univariate separation.**

That's surprising given what we might intuitively expect from a fraud dataset, but that's precisely why we inspect rather than assume.

### 🔒 VIABILITY GATE: PASS

But these decisions are now locked:

| Feature                | Current decision                              |
| ---------------------- | --------------------------------------------- |
| `txn_origin_country`   | ❌ Exclude. Strong synthetic target artifact  |
| `billing_country`      | ⚠️ Redundant with `bin_country`               |
| `bin_country`          | ⚠️ Likely weak                                |
| `currency`             | ⚠️ Weak marginal signal                       |
| `card_type`            | ⚠️ Weak marginal signal                       |
| `bank_name`            | ⚠️ Weak marginal signal                       |
| `session_time_seconds` | ⚠️ Very weak marginal signal                  |
| `failed_attempts`      | ⚠️ No marginal signal                         |
| `amount`               | ✅ Meaningful signal                          |
| `amount_in_usd`        | ✅ Meaningful signal                          |
| `card_bin`             | ✅ Potentially meaningful                     |
| `loyalty_tier`         | ✅ Meaningful                                 |
| `account_age_days`     | ✅ Meaningful                                 |
| `route`                | 🔍 Needs careful investigation                |

But these are still not final feature-selection decisions yet. We'll decide which features goes into the final processed dataset in the Feature Engineering Phase.

## 11. Temporal Fraud Pattern

Examine monthly transaction volume and fraud rate to assess temporal stability of the target.

In [20]:
monthly_fraud = (
    df.assign(month=df["transaction_date"].dt.to_period("M"))
    .groupby("month", observed=True)["is_fraud"]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "count": "transactions",
        "sum": "fraud_count",
        "mean": "fraud_rate"
    })
)

monthly_fraud["fraud_rate"] = monthly_fraud["fraud_rate"] * 100

monthly_fraud

,transactions,fraud_count,fraud_rate
month,,,
2025-12,517086,8332,1.611337
2026-01,516504,7810,1.512089
2026-02,467318,6816,1.458536
2026-03,516061,7355,1.425219
2026-04,499421,7456,1.492929
2026-05,483610,7231,1.495213


`December` is somewhat higher at `1.61%`. Then fraud rate gradually falls toward `~1.43%` by `March`, before settling around `~1.49%` in `April-May`. So, there is some temporal variation, but no dramatic regime shift.

Because `December` has `517k` transactions and still has a higher fraud rate, this isn't just a tiny-sample fluctuation. Likewise, `March` has `516k` transactions and the lowest rate.

So there is some genuine temporal structure in the generated data and this strengthens our argument for eventually using a time-aware validation strategy rather than randomly mixing all six months.

## 12. Route-Level Fraud Pattern

Assess route-level fraud rates alongside transaction volume to distinguish potentially meaningful route effects from sparse-category noise.

In [21]:
route_fraud = (
    df.groupby("route", observed=True)["is_fraud"]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "count": "transactions",
        "sum": "fraud_count",
        "mean": "fraud_rate"
    })
)

route_fraud["fraud_rate"] = route_fraud["fraud_rate"] * 100

route_fraud.sort_values(
    ["fraud_rate", "transactions"],
    ascending=[False, False]
).head(15)

,transactions,fraud_count,fraud_rate
route,,,
TPE-HKG,24,2,8.333333
DEL-SYD,85,5,5.882353
DXB-CGK,17,1,5.882353
CDG-OKA,18,1,5.555556
AMS-KIX,37,2,5.405405
JFK-MEL,40,2,5.000000
CGK-HAN,20,1,5.000000
MEL-OKA,223,11,4.932735
HND-MAD,42,2,4.761905


In [22]:
route_fraud.sort_values(
    ["fraud_rate", "transactions"],
    ascending=[False, False]
).iloc[-60: -50]

,transactions,fraud_count,fraud_rate
route,,,
NRT-OKA,342,1,0.292398
CGK-SIN,181,0,0.000000
OKA-SGN,170,0,0.000000
SUB-MAD,165,0,0.000000
SYD-BKK,159,0,0.000000
BKK-SUB,143,0,0.000000
HKG-FRA,128,0,0.000000
LAS-HND,128,0,0.000000
ORD-CDG,128,0,0.000000


`route` may genuinely contain signal, but the raw fraud rate is clearly affected by category size. Route is potentially useful, but naive route-level fraud rates are unstable for sparse routes.

| Finding                                            | Decision                                       |
| -------------------------------------------------- | ---------------------------------------------- |
| 1,558 routes                                       | High cardinality                               |
| Extreme fraud rates                                | Mostly associated with relatively small groups |
| Larger routes can still show elevated fraud        | Potential real signal                          |
| No obvious synthetic 0%/100% mechanism             | Good                                           |
| Raw route fraud rate is unstable for sparse routes | Handle carefully later                         |

## 13. Save Cleaned Analytical Dataset

Save the validated and memory-optimized analytical dataset as a Parquet checkpoint for the feature engineering phase.

In [23]:
from pathlib import Path

In [ ]:
project_root = Path.cwd().parent
output_path = project_root / "data" / "processed" / "airline_fraud_cleaned.parquet"

# Save the validated and memory-optimized DataFrame
df.to_parquet(output_path, engine="pyarrow", index=False)

print(f"Successfully saved to: {output_path.relative_to(project_root)} 🎉")

Successfully saved to: data\processed\airline_fraud_cleaned.parquet 🎉
